# 05 — Fila na L4: a unidade de bloco a 512 px e as sementes que faltam

## O que a L4 compra

A receita vencedora até aqui, medida na RTX 2060 (8 GB), é o
`configs/allsky/experiments/ceuv3res512/ceuv3res512_s42.yaml`: DINOv3 ViT-S/16+ a **512 px**
(a resolução dos frames em disco, 1.024 tokens por imagem), `image_only`, alvo no k\* com o céu
a 0,25, LR por camada 0,8 e `backbone_lr` 1e-5, jitter de exposição + C-Mixup, batch 16, 40
épocas de cosseno, monitor `val/kindex_mae`, **last** como checkpoint operacional. No teste
novo, com o **last**: DHI **18,0 W/m²**, kt-bal **0,669**; por bloco, macro-F1 **0,685** pela
classe reconstruída do k\* (`pred_sky_kt`) e **0,653** pela cabeça de céu (`pred_sky`), contra
0,659 da persistência do bloco anterior. Com o **best**: 18,1 / 0,672 / 0,695 / 0,674. Os dois
estimadores por bloco saem lado a lado na tabela desta fila (`block_kt_macro_f1` e
`block_macro_f1`); um número por bloco só se compara ao do mesmo estimador e do mesmo checkpoint.

O que 8 GB não deixam fazer, e 24 GB deixam:

1. **A unidade de bloco a 512 px.** `alignment.strategy: sensor_block` faz de cada amostra os
   4–5 frames da mesma linha do CR5000 — o suporte do próprio rótulo — e é o único mecanismo
   que move o teto: 0,73 por frame → 0,84+ por bloco. A 512 px um item são até cinco imagens
   de 1.024 tokens; batch 8 = 40 imagens por passo (~15 GB).
2. **Batch maior nas sementes a 512.** Duas sementes novas do vencedor (44 e 45, cosseno de
   cosseno de 150 épocas com parada por paciência 15 no `val/kindex_mae`) com batch 32 (~12 GB), para fechar em quatro sementes o intervalo
   de confiança do resultado.

Os três configs vivem no repositório, em `configs/allsky/experiments/l4/`, e são rodados
**como estão**: este notebook não escreve YAML derivado. O `amp` deles é `bf16` (a L4 é Ada);
a célula de hardware confere que a GPU atribuída tem bfloat16 antes de treinar.

| entrada | config | épocas | batch | estimativa na L4 |
|---|---|---|---|---|
| bloco | `l4/l4bloco512_s42.yaml` | até 150, paciência 15 | 8 blocos (40 frames) | ~8 min/época → ~5–8 h até parar, mais as três avaliações |
| semente 44 | `l4/l4v3res512_s44.yaml` | até 150, paciência 15 | 32 | ~5–7 h até parar |
| semente 45 | `l4/l4v3res512_s45.yaml` | até 150, paciência 15 | 32 | ~5–7 h até parar |

A base da estimativa é a cadência local: 14,6 min/época a 512 px com batch 16 na 2060
(`ceuv3res512_s42`, 40 épocas em 9 h 43), com a L4 a ~2× disso em bf16. A soma passa de
**24 h**, o teto do job: a terceira entrada pode não caber. Não é um problema — a fila é
**retomável**: cada braço concluído fica arquivado no bucket, e uma segunda execução deste
notebook pula o que já tem `eval-test-last/eval_metrics.json` lá e só roda o que falta.

## O que fica arquivado, e onde

Tudo sai da VM pelo bucket `gs://labmim-allsky-506901/runs/allsky-l4/`, espelhado a cada
5 min e ao fim de cada braço:

- `<nome>/` — `metrics.csv`/`metrics.json` do treino, os relatórios `eval-test` (best),
  `eval-val` (best) e `eval-test-last` (last), com `predictions.parquet`, `stratified.csv`,
  `confusion.csv` e `report.md`, e o YAML do config. Sem checkpoint (`archive` os exclui).
- `_live/` — o `metrics.csv` de cada braço época a época e `heartbeat.json` com a GPU: é o
  que diz, de fora, se a VM está treinando ou parada.
- `campanha_l4_parcial.csv` após cada braço; `campanha_l4.csv` e `campanha_l4_resumo.json`
  no fechamento.

## Antes de rodar

1. O bucket tem `colab/micrometeorology.bundle` (git bundle de `BRANCH`, com
   `configs/allsky/experiments/l4/` e este `_colab_runner`), `allsky-mm/bundle-iso-20260906.tar.gz`
   (dataset-iso de 512 px, 88 dias, com frames) e
   `dinov3/dinov3_vits16plus_pretrain_lvd1689m.pth`.
2. Template `labmim-l4` (g2-standard-8: 8 vCPU, 1× NVIDIA L4 24 GB). O `num_workers: 8` dos
   configs casa com as 8 vCPU.
3. Só Colab Enterprise: não há Drive nem fila remota aqui. O job termina sozinho ao fim do
   notebook.

## 1. Runtime e GPU

In [ ]:
import subprocess
import time

SESSION_START = time.time()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout)

## 2. Ambiente

A mesma célula do notebook 04: clona o repositório onde o `_colab_runner` mora, instala o
torch CUDA pelo backend que o driver da VM pede (`--torch-backend auto`) e **verifica**. No
Colab Enterprise (`VERTEX_PRODUCT=COLAB_ENTERPRISE`) o repositório vem do `git bundle` no
bucket; fora dele, do GitHub.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/Bruno-Mascarenhas/micrometeorology.git"
BRANCH = (
    "condicao-do-ceu-multitarefa"  # a branch que carrega os configs de l4/ e este _colab_runner
)
ON_VERTEX = os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE" or not os.path.isdir("/content")
BUCKET = "gs://labmim-allsky-506901"
BASE = str(Path.home()) if ON_VERTEX else "/content"
WORKDIR = f"{BASE}/micrometeorology"

if not os.path.exists(WORKDIR):
    if ON_VERTEX:
        bundle = f"{BASE}/micrometeorology.bundle"
        subprocess.run(
            ["gcloud", "storage", "cp", f"{BUCKET}/colab/micrometeorology.bundle", bundle],
            check=True,
        )
        subprocess.run(["git", "clone", "-b", BRANCH, bundle, WORKDIR], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORKDIR], check=True)
if not os.path.isdir(f"{WORKDIR}/configs/allsky/experiments/l4"):
    raise RuntimeError(
        f"a branch {BRANCH} do bundle nao carrega configs/allsky/experiments/l4/ — refaca o bundle"
    )
subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.14"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "venv", "--python", "3.14", ".venv"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "sync", "--locked", "--extra", "allsky"], cwd=WORKDIR, check=True)
subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--python",
        ".venv/bin/python",
        "--reinstall",
        "--torch-backend",
        "auto",
        "torch==2.13.0",
    ],
    cwd=WORKDIR,
    check=True,
)

PY = f"{WORKDIR}/.venv/bin/python"
os.environ["PATH"] = f"{WORKDIR}/.venv/bin:" + os.environ["PATH"]

# O DINOv3 nao vem pelo torch.hub (o hubconf arrasta torchmetrics/omegaconf/submitit):
# o pacote importa hub/backbones.py direto do clone, apontado por ALLSKY_DINOV3_REPO.
DINOV3_REPO_DIR = f"{BASE}/dinov3"
if not os.path.exists(DINOV3_REPO_DIR):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/facebookresearch/dinov3",
            DINOV3_REPO_DIR,
        ],
        check=True,
    )
os.environ["ALLSKY_DINOV3_REPO"] = DINOV3_REPO_DIR
sys.path.insert(0, f"{WORKDIR}/notebooks/colab")

verify = subprocess.run(
    [PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True,
    text=True,
    check=False,
)
print(verify.stdout)
if "True" not in verify.stdout:
    raise RuntimeError("torch sem CUDA — pare e reinstale antes de treinar")

import _colab_runner as runner  # noqa: E402

if not hasattr(runner, "score_by_sensor_block") or not hasattr(runner, "mirror_once"):
    raise RuntimeError(
        f"o _colab_runner de {BRANCH} nao tem score_by_sensor_block/mirror_once — "
        "aponte BRANCH para uma branch que os carregue"
    )

## 3. Hardware

O probe roda no interpretador do venv. Os três configs declaram `amp: bf16`, então uma GPU
sem bfloat16 (T4, Turing) pára aqui — antes de baixar o bundle de 1,4 GB e desempacotá-lo, e
não na primeira época.

Memória esperada na L4 (24 GB), a partir do medido na 2060 (16 imagens de 512 px ≈ 6 GB
em fp16 com o ViT-S/16+ inteiro descongelado):

| entrada | imagens por passo | estimativa |
|---|---|---|
| bloco, batch 8 × até 5 frames | 40 | ~15 GB |
| sementes, batch 32 | 32 | ~12 GB |

In [ ]:
import json

probe = subprocess.run(
    [
        PY,
        "-c",
        'import json, sys; sys.path.insert(0, "' + WORKDIR + '/notebooks/colab"); '
        "import _colab_runner as r; print(json.dumps(r.probe_accelerator()))",
    ],
    capture_output=True,
    text=True,
    check=True,
)
HW = json.loads(probe.stdout.strip().splitlines()[-1])
print(HW)
if HW["amp_dtype"] != "bf16":
    raise RuntimeError(
        f"{HW['name']} sem bfloat16: os configs de l4/ declaram amp bf16 — peca o template labmim-l4"
    )
if HW["cpus"] < 8:
    print(f"ATENCAO: {HW['cpus']} vCPU para num_workers: 8 dos configs — o loader vai disputar CPU")

## 4. Dados e artefatos

Só bucket. O bundle e os pesos do DINOv3 vêm para o disco da VM; `stage_bundle` desempacota
e roda `validate-dataset`. O arquivo de cada braço é escrito em `ARTIFACTS` (disco local) e
**espelhado** para `gs://…/runs/allsky-l4`.

**Retomada.** Antes de qualquer coisa o conteúdo já arquivado no bucket é trazido de volta
para `ARTIFACTS`: é por ele que a fila (seção 5) sabe o que já está feito. Na primeira
execução o prefixo não existe (`gcloud storage ls` falha) e o `rsync` de volta falha junto —
esperado; numa retomada, com o prefixo listado, uma falha aqui significaria repetir horas de
treino, por isso ela pára o notebook antes de treinar.

**O link do dataset.** Os configs do repositório apontam `data_root:
output/allsky-mm/dataset-iso-20260906`, relativo ao repositório; um link simbólico desse
caminho para a raiz do bundle desempacotado deixa-os funcionar sem reescrita — e o `chdir`
para `WORKDIR` é o que faz `output_dir` (também relativo) e o `allsky train` que o `run_experiment`
lança resolverem contra o mesmo lugar.

In [ ]:
import os
from pathlib import Path

if not ON_VERTEX:
    raise RuntimeError(
        "este notebook so roda no Colab Enterprise: dado e arquivo vivem no bucket, nao ha Drive"
    )

STORE = f"{BASE}/labmim"
BUNDLE_REL = "allsky-mm/bundle-iso-20260906.tar.gz"
WEIGHTS_REL = "dinov3/dinov3_vits16plus_pretrain_lvd1689m.pth"
for rel in (BUNDLE_REL, WEIGHTS_REL):
    os.makedirs(f"{STORE}/{os.path.dirname(rel)}", exist_ok=True)
    subprocess.run(
        ["gcloud", "storage", "cp", "-n", f"{BUCKET}/{rel}", f"{STORE}/{rel}"], check=True
    )

BUNDLE = f"{STORE}/{BUNDLE_REL}"
DATA = f"{BASE}/allsky-mm"
ARTIFACTS = f"{STORE}/runs/allsky-l4"
REMOTE_ARTIFACTS = f"{BUCKET}/runs/allsky-l4"
os.environ["ALLSKY_DINOV3_WEIGHTS"] = f"{STORE}/{WEIGHTS_REL}"

os.makedirs(ARTIFACTS, exist_ok=True)
listing = subprocess.run(
    ["gcloud", "storage", "ls", REMOTE_ARTIFACTS], capture_output=True, text=True, check=False
)
pulled = runner.mirror_once([(REMOTE_ARTIFACTS, ARTIFACTS)])
if pulled and listing.returncode == 0:
    raise RuntimeError(
        f"{REMOTE_ARTIFACTS} existe mas o rsync de volta falhou: sem o arquivo a fila retreinaria tudo"
    )
print(
    "arquivo trazido do bucket:",
    "ok" if listing.returncode == 0 else "prefixo ainda nao existe, primeira execucao",
)
print("ja arquivado:", sorted(p.name for p in Path(ARTIFACTS).iterdir()) or "nada")
MIRROR = [(ARTIFACTS, REMOTE_ARTIFACTS)]
print("espelho inicial:", runner.mirror_once(MIRROR) or "ok")
runner.start_mirror(MIRROR)

ROOT = runner.stage_bundle(BUNDLE, DATA, python=PY)
for required in ("manifest.parquet", "splits.json", "frames"):
    if not (Path(ROOT) / required).exists():
        raise RuntimeError(
            f"{ROOT} sem {required}: o bundle nao e o dataset-iso-20260906 com frames"
        )

DATASET_LINK = Path(WORKDIR) / "output/allsky-mm/dataset-iso-20260906"
DATASET_LINK.parent.mkdir(parents=True, exist_ok=True)
if DATASET_LINK.is_symlink():
    DATASET_LINK.unlink()
elif DATASET_LINK.exists():
    raise RuntimeError(f"{DATASET_LINK} existe e nao e um link: nao vou sobrescrever")
DATASET_LINK.symlink_to(ROOT, target_is_directory=True)
os.chdir(WORKDIR)
print(f"{DATASET_LINK} -> {os.readlink(DATASET_LINK)}; cwd {os.getcwd()}")

## 5. A fila

Lista **fixa**, na ordem em que o tempo importa: o bloco primeiro (é a hipótese que move o
teto), depois as duas sementes. Para cada entrada:

1. se `ARTIFACTS/<nome>/eval-test-last/eval_metrics.json` já existe, é retomada: nada é
   treinado, as linhas vêm dos relatórios arquivados (só dos que existem);
2. senão `run_experiment` treina e avalia **best no teste**, **best na validação** e
   **last no teste** — nesta ordem, para que `eval-test-last` existir signifique braço completo;
3. pontuação **por bloco** de cada relatório com `score_by_sensor_block` — para o braço de
   bloco isso é idêntico ao relatório (a avaliação já serve um item por linha do logger);
4. `archive` para `ARTIFACTS`, linha na tabela, `campanha_l4_parcial.csv` e `mirror_once`.

O resumo por braço traz a difusa (RMSE/MAE/MBE), o céu por Kt reconstruído (acurácia
balanceada, macro-F1, F1 da parcialmente-clara) e o macro-F1 por bloco em dois estimadores,
a cabeça de céu (`pred_sky`, coluna `block_macro_f1`) e a classe reconstruída do k\*
(`pred_sky_kt`, coluna `block_kt_macro_f1`), contra a persistência do bloco anterior. O critério
de escolha continua sendo a **validação**; o teste está ali para ser comparado com o
`ceuv3res512_s42` no mesmo checkpoint e estimador: last 18,0 / 0,669 / bloco 0,685 (k\*) e
0,653 (céu); best 18,1 / 0,672 / 0,695 e 0,674; persistência 0,659.

Um braço arquivado pela metade (sem `eval-test-last`) é retreinado do zero e reavaliado nas
três combinações, para a linha ter um só checkpoint de origem: o arquivo só é consultado
quando o braço está completo.

In [ ]:
from pathlib import Path

import pandas as pd

FILA = [
    "configs/allsky/experiments/l4/l4bloco512_s42.yaml",
    "configs/allsky/experiments/l4/l4v3res512_s44.yaml",
    "configs/allsky/experiments/l4/l4v3res512_s45.yaml",
]
missing = [rel for rel in FILA if not (Path(WORKDIR) / rel).is_file()]
if missing:
    raise RuntimeError(f"{missing}: a branch {BRANCH} do bundle nao carrega os configs de l4/")

OUT = Path(WORKDIR) / "output/allsky-mm/experiments/l4"
EVALUATIONS = (("test", "best", ""), ("val", "best", "_val"), ("test", "last", "_last"))
REPORT_DIRS = {"": "eval-test", "_val": "eval-val", "_last": "eval-test-last"}
KEYS = (
    "rmse",
    "mae",
    "mbe",
    "sky_kt_balanced_accuracy",
    "sky_kt_macro_f1",
    "sky_kt_f1_partly_cloudy_clear",
    "sky_balanced_accuracy",
    "sky_macro_f1",
)
rows = []
runner.start_live_sync(OUT, Path(ARTIFACTS) / runner.LIVE_DIR)


def archived_report(name, report):
    """The archived metrics file of one report, whether or not it exists."""
    return Path(ARTIFACTS) / name / report / "eval_metrics.json"


def fmt(value, digits=2, sign=False):
    """A number for the summary line, or a dash where the row has none."""
    if value is None or pd.isna(value):
        return "—"
    return f"{value:+.{digits}f}" if sign else f"{value:.{digits}f}"


def score_blocks(row, name):
    """Per-block scores of every report of *name*, local run first, archive second."""
    for tag, report in REPORT_DIRS.items():
        parquet = OUT / name / "run" / report / "predictions.parquet"
        if not parquet.exists():
            parquet = Path(ARTIFACTS) / name / report / "predictions.parquet"
        if not parquet.exists():
            continue
        predictions = pd.read_parquet(parquet)
        block = runner.score_by_sensor_block(predictions, n_bootstrap=200)
        row[f"block_macro_f1{tag}"] = block["sky"]["macro_f1"]
        row[f"block_persistence_f1{tag}"] = block["sky_persistence_previous_block"]["macro_f1"]
        row[f"block_rmse{tag}"] = block["dhi"]["rmse"]
        if "pred_sky_kt" in predictions.columns:
            by_kt = runner.score_by_sensor_block(
                predictions, sky=("obs_sky", "pred_sky_kt"), n_bootstrap=200
            )
            row[f"block_kt_macro_f1{tag}"] = by_kt["sky"]["macro_f1"]


def resumo(row):
    """One line per arm: diffuse, sky by reconstructed Kt, and the per-block scores of both estimators."""
    print(
        f"{row['name']:<16} {row.get('status')}  "
        f"DHI rmse/mae/mbe {fmt(row.get('rmse'))}/{fmt(row.get('mae'))}/{fmt(row.get('mbe'), sign=True)} "
        f"(last {fmt(row.get('rmse_last'))}/{fmt(row.get('mae_last'))}/{fmt(row.get('mbe_last'), sign=True)})  "
        f"sky_kt bal {fmt(row.get('sky_kt_balanced_accuracy'), 3)} F1 {fmt(row.get('sky_kt_macro_f1'), 3)} "
        f"parc-clara {fmt(row.get('sky_kt_f1_partly_cloudy_clear'), 3)} "
        f"(last bal {fmt(row.get('sky_kt_balanced_accuracy_last'), 3)})  "
        f"por bloco F1 ceu {fmt(row.get('block_macro_f1'), 3)} "
        f"(last {fmt(row.get('block_macro_f1_last'), 3)}, val {fmt(row.get('block_macro_f1_val'), 3)}) "
        f"k* {fmt(row.get('block_kt_macro_f1'), 3)} "
        f"(last {fmt(row.get('block_kt_macro_f1_last'), 3)}, val {fmt(row.get('block_kt_macro_f1_val'), 3)}); "
        f"persistencia {fmt(row.get('block_persistence_f1'), 3)}; RMSE {fmt(row.get('block_rmse'))}"
    )


def braco(config_rel):
    """Run (or resume) one queue entry through train, three evaluations, block scores and archive."""
    config = Path(WORKDIR) / config_rel
    name = config.stem
    resumed = archived_report(name, "eval-test-last").exists()
    if resumed:
        print(f"{name}: ja arquivado em {ARTIFACTS} — retomada, nada a treinar")
    row = {"name": name, "config": config_rel, "status": "archived" if resumed else "pending"}
    for split, checkpoint, tag in EVALUATIONS:
        report = REPORT_DIRS[tag]
        if resumed and not archived_report(name, report).exists():
            print(f"  {report}: nao arquivado, a coluna fica vazia")
            continue
        result = runner.run_experiment(
            config,
            python=PY,
            split=split,
            checkpoint=checkpoint,
            archive_dir=ARTIFACTS if resumed else None,
        )
        if result.get("status") not in ("ok", "archived"):
            row["status"] = result.get("status")
            row["error"] = str(result.get("error"))[-800:]
            print(f"  {report}: {row['status']}\n{row['error']}")
            break
        if tag == "":
            row.update(
                {key: value for key, value in result.items() if key not in ("config", "checkpoint")}
            )
        else:
            row.update({f"{key}{tag}": result.get(key) for key in KEYS})
    score_blocks(row, name)
    print(" ", runner.archive(str(OUT / name), ARTIFACTS, config=config))
    rows.append(row)
    pd.DataFrame(rows).to_csv(f"{ARTIFACTS}/campanha_l4_parcial.csv", index=False)
    print("  espelho:", runner.mirror_once(MIRROR) or "ok")
    resumo(row)
    return row


for entry in FILA:
    started = time.time()
    braco(entry)
    print(
        f"  {(time.time() - started) / 3600:.1f} h nesta entrada; {(time.time() - SESSION_START) / 3600:.1f} h de sessao"
    )

## 6. Fechamento

Grava o índice da campanha e o resumo, e espelha uma última vez. No Colab Enterprise não há
`runtime.unassign`: o job termina quando o notebook termina.

In [ ]:
frame = pd.DataFrame(rows)
frame.to_csv(f"{ARTIFACTS}/campanha_l4.csv", index=False)
summary = {
    "hardware": HW,
    "fila": FILA,
    "n_runs": len(rows),
    "session_hours": round((time.time() - SESSION_START) / 3600, 2),
    "rows": rows,
}
with open(f"{ARTIFACTS}/campanha_l4_resumo.json", "w") as handle:
    json.dump(summary, handle, indent=2, default=str)
for row in rows:
    resumo(row)
print(frame.to_string())
print("espelho final:", runner.mirror_once(MIRROR) or "ok")
print("artefatos em", ARTIFACTS, "->", REMOTE_ARTIFACTS)